In [0]:
%run ../init_framework

In [0]:
# read from bronze cdf. startingVersion 1 is req for the very first pipeline run.
df_bronze_customers = (spark.readStream
    .format("delta")
    .option("readChangeFeed", "true") 
    .option("startingVersion", 1) 
    .table(CUSTOMERS_BRONZE))

In [0]:
CUSTOMER_RENAME_MAP = {
    "annual_inc": "annual_income",
    "addr_state": "address_state",
    "zip_code": "address_zipcode",
    "country": "address_country",
    "tot_hi_cred_lim": "total_high_credit_limit", 
    "annual_inc_joint": "joint_annual_income"
}

# align names to target silver ddl
df_renamed_customers = standardize_column_names(df_bronze_customers, CUSTOMER_RENAME_MAP, strict=True)

In [0]:
# inject standard audit cols
df_with_metadata = apply_silver_metadata(df_renamed_customers)

In [0]:
# if upstream guarantees no exact row dupes later
df_distinct = df_with_metadata.distinct()

In [0]:
# drop records where annual income is null/unknown (critical for risk model)
col_ls = ["annual_income"]
df_valid_customers = drop_critical_nulls(df_distinct, col_ls)

In [0]:
from pyspark.sql.functions import col, regexp_replace

# strip non-digits and emp_length cast as INT
df_emplength_cleaned = df_valid_customers.withColumn(
    "emp_length", regexp_replace(col("emp_length"), r"\D+", "").cast("int")
)

In [0]:
# default missing employment duration to -1 for downstream 
df_emplength_notnull = df_emplength_cleaned.fillna(-1, subset=["emp_length"])

In [0]:
# enforce strict 2-char state codes, fallback to NA
df_state_cleaned = df_emplength_notnull.withColumn(
    "address_state",
    when(
        length(trim(col("address_state"))) == 2, 
        upper(trim(col("address_state")))
    )
    .otherwise(lit("NA")) 
)

In [0]:
# final dedup on PK to prevent merge conflicts in the foreachBatch
df_customers_final = df_state_cleaned.dropDuplicates(["member_id"])

In [0]:
# setup target shuffle partitions based on our cluster size (8 cores)
spark.conf.set("spark.sql.shuffle.partitions", "16")

def upsert_customers_to_silver(microBatchDF, batchId):
    # stateless foreachbatch merge wrapper
    spark_session = microBatchDF.sparkSession

    # register view for spark sql engine
    microBatchDF.createOrReplaceTempView("customers_batch_updates")

    merge_query = f"""
    MERGE INTO {CUSTOMERS_SILVER} AS target
        USING customers_batch_updates AS source
        ON target.member_id = source.member_id
        WHEN MATCHED AND source._bronze_version > target._bronze_version THEN
          UPDATE SET
            target.emp_title = source.emp_title,
            target.emp_length = source.emp_length,
            target.home_ownership = source.home_ownership,
            target.annual_income = source.annual_income,
            target.address_state = source.address_state,
            target.address_zipcode = source.address_zipcode,
            target.address_country = source.address_country,
            target.grade = source.grade,
            target.sub_grade = source.sub_grade,
            target.verification_status = source.verification_status,
            target.total_high_credit_limit = source.total_high_credit_limit,
            target.application_type = source.application_type,
            target.joint_annual_income = source.joint_annual_income,
            target.verification_status_joint = source.verification_status_joint,
            target._ingested_at = source._ingested_at,
            target._source_file = source._source_file,
            target._bronze_version = source._bronze_version,
            target._updated_at = source._updated_at
        WHEN NOT MATCHED THEN
          INSERT (
            member_id, emp_title, emp_length, home_ownership, annual_income, 
            address_state, address_zipcode, address_country, grade, sub_grade, 
            verification_status, total_high_credit_limit, application_type, 
            joint_annual_income, verification_status_joint, 
            _ingested_at, _source_file, _bronze_version, _updated_at
          )
          VALUES (
            source.member_id, source.emp_title, source.emp_length, source.home_ownership, source.annual_income, 
            source.address_state, source.address_zipcode, source.address_country, source.grade, source.sub_grade, 
            source.verification_status, source.total_high_credit_limit, source.application_type, 
            source.joint_annual_income, source.verification_status_joint, 
            source._ingested_at, source._source_file, source._bronze_version, source._updated_at
          )
    """
    spark_session.sql(merge_query)

 
# --- init write stream ---
query_customers = (
    df_customers_final.writeStream
    .outputMode("append") 
    .option("checkpointLocation", SILVER_CHECKPOINT_CUSTOMERS)
    .trigger(availableNow=True)
    .foreachBatch(upsert_customers_to_silver)
    .start()
)

# block cluster exit until this stream finishes
query_customers.awaitTermination()

In [0]:
%sql
select count(1) from lending_risk.silver.customers
-- 1505607